# 🤖 Josh Kenya LLM - Google Colab Training Notebook

## Train a 50M Parameter Language Model on Free GPU

This notebook sets up and trains Josh Kenya LLM on Google Colab. The entire process takes about 1.5-2 hours on a free T4 GPU.

### Setup Instructions:
1. Click **Runtime** → **Change runtime type** → Select **GPU (T4)**
2. Run each cell in order
3. Download your trained model when done!

## Step 1: Install Dependencies

In [ ]:
# Install required packages
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q datasets transformers tqdm pyyaml flask flask-cors
print("✓ Dependencies installed")

## Step 2: Clone Repository from Correct Branch

In [ ]:
# Clone the repository from llm-project-setup branch
!git clone -b llm-project-setup https://github.com/aipulse54-svg/Llm_kenya.git
%cd Llm_kenya

# Verify all files are present
import os
required_files = ['config.yaml', 'requirements.txt', 'src/model_architecture.py', 'src/trainer.py', 'src/inference.py', 'src/tokenizer.py']
print("\nVerifying files:")
for file in required_files:
    if os.path.exists(file):
        print(f"  ✓ {file}")
    else:
        print(f"  ✗ {file} (MISSING)")

## Step 3: Setup Python Path and Check GPU

In [ ]:
import sys
import os

# Add src to path
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))

# Check GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print("\n✓ GPU is ready!")
else:
    print("\n⚠️  GPU not available!")
    print("Please go to Runtime → Change runtime type and select GPU (T4)")

## Step 4: Verify Model Architecture

In [ ]:
# Import model and config
from model_architecture import JoshKenyaLLM
import yaml

# Load config
with open('config.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Create model to check architecture
model = JoshKenyaLLM(config['model']).to('cpu')  # Use CPU to just check architecture
num_params = model.get_num_params()

print("="*60)
print("Josh Kenya LLM Model Architecture".center(60))
print("="*60)
print(f"Total Parameters: {num_params:,} ({num_params/1e6:.1f}M)")
print(f"\nModel Configuration:")
print(f"  Hidden Size: {config['model']['hidden_size']}")
print(f"  Num Layers: {config['model']['num_hidden_layers']}")
print(f"  Attention Heads: {config['model']['num_attention_heads']}")
print(f"  Intermediate Size: {config['model']['intermediate_size']}")
print(f"  Vocab Size: {config['model']['vocab_size']}")
print(f"  Max Position Embeddings: {config['model']['max_position_embeddings']}")
print(f"  Model Name: Josh Kenya")
print("="*60)

# Delete model from memory
del model

## Step 5: Customize Training Config (Optional)

Modify settings below if needed. For free Colab T4 GPU with ~15GB VRAM:

In [ ]:
# Optimize for Colab free T4 GPU
config['training']['batch_size'] = 16  # Reduced from 32 for free T4
config['training']['num_epochs'] = 1   # Start with 1 epoch (can increase to 2-3)
config['training']['learning_rate'] = 5e-4
config['model']['max_position_embeddings'] = 1024  # Reduced from 2048
config['training']['num_workers'] = 2

# Save updated config
with open('config.yaml', 'w') as f:
    yaml.dump(config, f)

print("✓ Training config optimized for Colab")
print(f"\nTraining Settings:")
print(f"  Batch Size: {config['training']['batch_size']}")
print(f"  Learning Rate: {config['training']['learning_rate']}")
print(f"  Num Epochs: {config['training']['num_epochs']}")
print(f"  Max Seq Length: {config['model']['max_position_embeddings']}")
print(f"  Num Workers: {config['training']['num_workers']}")
print(f"\nEstimated time: ~20-30 minutes per epoch on free T4")

## Step 6: Start Training 🚀

This will train the model on WikiText-2 dataset. **This may take 20-30 minutes.**

In [ ]:
from trainer import LLMTrainer
import torch

print("\n" + "="*60)
print("Josh Kenya LLM - Training Pipeline".center(60))
print("="*60 + "\n")

try:
    # Clear GPU cache
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    # Initialize trainer
    trainer = LLMTrainer(config_path='config.yaml', device='cuda')
    
    print(f"✓ Model initialized with {trainer.model.get_num_params():,} parameters")
    print(f"✓ Using device: {trainer.device}")
    print(f"\nStarting training...\n")
    
    # Start training
    trainer.train()
    
    print("\n" + "="*60)
    print("✓ Training completed successfully!".center(60))
    print("="*60)
    print(f"\n✓ Model saved to: models/josh_kenya_lm.pt")
    print(f"✓ Config saved to: models/config.yaml")
    print(f"✓ Checkpoints in: checkpoints/")
    
except KeyboardInterrupt:
    print("\n\nTraining interrupted by user")
except Exception as e:
    print(f"\nError during training: {e}")
    import traceback
    traceback.print_exc()

## Step 7: Verify Model Files

In [ ]:
import os
from pathlib import Path

print("Model files:")
if os.path.exists('models'):
    for file in os.listdir('models'):
        path = os.path.join('models', file)
        if os.path.isfile(path):
            size = os.path.getsize(path) / 1e6  # Size in MB
            print(f"  ✓ {file} ({size:.1f} MB)")
else:
    print("  No models directory found")

print("\nCheckpoint files:")
if os.path.exists('checkpoints'):
    checkpoint_files = sorted(os.listdir('checkpoints'))
    for file in checkpoint_files:
        path = os.path.join('checkpoints', file)
        if os.path.isfile(path):
            size = os.path.getsize(path) / 1e6
            print(f"  ✓ {file} ({size:.1f} MB)")
else:
    print("  No checkpoints directory found")

## Step 8: Test Inference

In [ ]:
from inference import JoshKenyaInference
import torch

print("Loading trained model...")
try:
    # Clear GPU cache
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    model = JoshKenyaInference(model_dir='models', device='cuda')
    
    # Show model info
    info = model.get_model_info()
    print(f"\nModel Information:")
    print(f"  Name: {info['name']}")
    print(f"  Parameters: {info['parameters_millions']}")
    print(f"  Device: {info['device']}")
    print(f"  Memory: {info['memory_size']}/{info['memory_max']} turns")
    
    # Test generation
    print(f"\n{'='*60}")
    print("Testing model generation...".center(60))
    print('='*60)
    
    question = "What is artificial intelligence?"
    print(f"\nYou: {question}")
    print(f"\nJosh Kenya: ", end="", flush=True)
    
    response = model.answer_question(question)
    print(response)
    
    print(f"\n{'='*60}")
    print("✓ Model is working!".center(60))
    print('='*60)
    
except Exception as e:
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()

## Step 9: Download Trained Model

In [ ]:
# Create a zip file with the trained model
import shutil
import os

print("Creating download package...")

# Zip the models directory
if os.path.exists('models'):
    shutil.make_archive('josh_kenya_trained_model', 'zip', '.', 'models')
    size = os.path.getsize('josh_kenya_trained_model.zip') / 1e6
    print(f"✓ Created: josh_kenya_trained_model.zip ({size:.1f} MB)")
    print("\n📥 Download Instructions:")
    print("  1. Click the Files icon on the left sidebar")
    print("  2. Find 'josh_kenya_trained_model.zip'")
    print("  3. Right-click → Download")
    print("\n📋 To use the trained model locally:")
    print("  1. Download josh_kenya_trained_model.zip")
    print("  2. Unzip it in your project directory")
    print("  3. Run: python src/app.py")
    print("  4. Open: http://localhost:5000")
else:
    print("⚠️  Models directory not found. Train the model first.")

## Step 10: Multi-Turn Chat Test

In [ ]:
# Multi-turn conversation test
from inference import JoshKenyaInference
import torch

print("Testing multi-turn conversation memory...\n")
try:
    # Clear GPU cache
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    model = JoshKenyaInference(model_dir='models', device='cuda')
    
    # Test a conversation
    questions = [
        "What is machine learning?",
        "Can you explain neural networks?",
        "How are transformers different?"
    ]
    
    print(f"{'='*60}")
    print("Josh Kenya LLM - Multi-Turn Chat Test".center(60))
    print('='*60)
    
    for i, question in enumerate(questions, 1):
        print(f"\n[Turn {i}]")
        print(f"You: {question}")
        response = model.answer_question(question)
        print(f"Josh Kenya: {response[:100]}..." if len(response) > 100 else f"Josh Kenya: {response}")
        print("-" * 60)
    
    # Show conversation history
    history = model.memory.get_history()
    print(f"\nConversation memory: {len(history)} turns stored")
    print(f"Memory capacity: {model.memory.memory_size} turns")
    
    print(f"\n{'='*60}")
    print("✓ Multi-turn conversation working!".center(60))
    print('='*60)
    
except Exception as e:
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()

## Summary & Next Steps

### ✅ What you've accomplished:

1. **✓ Trained a 50M parameter LLM** on WikiText-2 dataset
2. **✓ Model saved** with all parameters and configuration
3. **✓ Tested inference** - model can answer questions and remember context
4. **✓ Ready to download** - use locally with web UI

### 📥 Download Your Model

1. Click **Files** (folder icon) on the left
2. Download `josh_kenya_trained_model.zip`

### 💻 Use Locally

```bash
# Unzip the model
unzip josh_kenya_trained_model.zip

# Start the web UI
python src/app.py

# Open browser
http://localhost:5000
```

### 🚀 Features

- **50M Parameters** - Efficient and trainable on free GPUs
- **Long Memory** - Remembers up to 10 conversation turns
- **Web UI** - Beautiful chat interface
- **Fast Inference** - GPU-accelerated generation
- **Self-Identifying** - Model knows its name: "Josh Kenya"

### 📚 Training Tips

- **Out of memory?** Reduce `batch_size` to 8
- **Want better results?** Increase `num_epochs` to 2-3
- **Need faster training?** Reduce `max_position_embeddings` to 512
- **Want to fine-tune?** Modify the dataset loader in `src/trainer.py`

---

**Congratulations on training Josh Kenya LLM! 🎉**

Your model is ready for production use. Enjoy!